**Import libraries**

In [ ]:
import pandas as pd

**read Artifact**

In [ ]:
ml_table = pd.read_parquet('ml_orders_dataset.parquet')

**Check order status and missing delivery dates before labeling**

In [ ]:
# orders with no delivered date were never delivered (canceled, lost, still in transit, etc.)
print("Order status counts:")
print(ml_table['order_status'].value_counts())

print("\nMissing order_delivered_customer_date:", ml_table['order_delivered_customer_date'].isna().sum())

**Drop orders that were never delivered**

"Late" vs "on time" only makes sense for an order that actually arrived. Comparing NaT to a date silently returns False in pandas, which would wrongly label undelivered orders as on time. So these rows are removed before building the label.

In [ ]:
before_rows = len(ml_table)

ml_table = ml_table[ml_table['order_delivered_customer_date'].notna()].copy()

print(f"Rows before: {before_rows}")
print(f"Rows after dropping undelivered orders: {len(ml_table)}")
print(f"Rows dropped: {before_rows - len(ml_table)}")

**Build the label**

In [ ]:
# label is 1 if the order arrived after the estimated delivery date, 0 otherwise
ml_table['is_late'] = (ml_table['order_delivered_customer_date'] > ml_table['order_estimated_delivery_date']).astype(int)

**Check the label**

In [ ]:
check_cols = ['order_delivered_customer_date', 'order_estimated_delivery_date', 'is_late']
print(ml_table[check_cols].head(50))

**Check a few edge cases specifically: orders delivered right around the estimated date**

In [ ]:
# gap in days between actual and estimated delivery, useful to sanity check borderline cases
ml_table['delivery_gap_days'] = (
    ml_table['order_delivered_customer_date'] - ml_table['order_estimated_delivery_date']
).dt.total_seconds() / 86400

# orders delivered within +/- 1 day of the estimate, to visually confirm the label boundary
borderline = ml_table[ml_table['delivery_gap_days'].abs() <= 1]
print(borderline[check_cols + ['delivery_gap_days']].head(10))

ml_table = ml_table.drop(columns=['delivery_gap_days'])

**Class distribution**

In [ ]:
print("\n is late value counts:")
print(ml_table['is_late'].value_counts())

print("\n late percentage:")
print(ml_table['is_late'].value_counts(normalize=True) * 100)

# imbalance ratio: on-time count divided by late count
counts = ml_table['is_late'].value_counts()
ratio = counts[0] / counts[1]
print(f"\nImbalance ratio (on-time : late): {ratio:.1f} : 1")

Yes, there is a class imbalance problem; there is a clear bias towards the On Time class.

See the computed ratio above for the exact imbalance (recomputed after removing undelivered orders).

**Artifact:ML labeled table**

In [ ]:
ml_table.to_parquet('ml_orders_labeled.parquet', index=False)